# Diet Data Enhancement Benchmark I

This TRE notebook benchmarks whether enhanced diet representations improve phenotype prediction compared with:

1. age + sex only,
2. age + sex + base nutrients,
3. age + sex + NutriMatch nutrients,
4. age + sex + each enhanced Diet Data Enhancement feature set.

The first section follows the NutriMatch paper's prediction design as closely as possible with the HPP TRE tables available here: hierarchical feature arms, fivefold cross-validation, and paper-aligned targets such as body-fat indices, waist, serum folate, CGM traits, blood biomarkers, and obesity/overweight-like outcomes. The paper used LightGBM; this notebook intentionally uses faster sklearn models and runs both Ridge and Random Forest for every section.

Later sections test new Diet Data Enhancement benchmark domains:

- gut microbiome features from diet,
- Nightingale metabolomics from diet,
- oral microbiome features from diet,
- CGM/post-meal glucose proxy traits from diet,
- peripheral vascular health from diet.

All intermediate diet X matrices and target matrices are cached. Leave the `FORCE_REBUILD_*` flags as `False` to avoid repeating the expensive extraction steps.


## 0. Imports And Global Settings


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys
import re
import json
import gc
import time
import warnings
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from scipy.stats import pearsonr
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, RidgeClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelBinarizer, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 240)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
N_SPLITS = 5
RF_N_ESTIMATORS = 250
MAX_FEATURES_PER_ARM = 700
MAX_TARGETS_PER_SECTION = 30
MAX_CLASSIFICATION_TARGETS_PER_SECTION = 12
MIN_N_PER_TARGET = 80
MIN_CLASS_COUNT = 25

FORCE_REBUILD_X = False
FORCE_REBUILD_TARGETS = False
RUN_CLASSIFICATION_DERIVATIVES = True

# Default notebook mode is plotting/loading only.
# The background runner sets DDE_RUN_TRAINING=1 so the expensive training runs outside the browser session.
RUN_TRAINING = os.environ.get("DDE_RUN_TRAINING", "0") == "1"

print("Notebook settings loaded.")
print("RUN_TRAINING:", RUN_TRAINING)


## 1. Paths, Feature Sets, And Caches

The notebook first looks for the TRE project path. If you run it from inside the Diet Data Enhancement repo, it uses the current directory. The cached participant X files are reused from the CVD experiment when present:

`downstream_analysis/tasks/cvd/outputs/<feature_set>/X_<feature_set>_participant.csv`


In [ ]:
PROJECT_ROOT = Path.cwd()
TRE_PROJECT_ROOT = Path("/home/ec2-user/studies/Diet_Data_Enhancement_Project/Diet_Data_Enhancement_TRE")
if not (PROJECT_ROOT / "downstream_analysis").exists() and TRE_PROJECT_ROOT.exists():
    PROJECT_ROOT = TRE_PROJECT_ROOT

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_STEM = "diet_data_enhancement_benchmark_I"
TRE_INPUTS = PROJECT_ROOT / "tre_inputs"
CVD_OUTPUTS = PROJECT_ROOT / "downstream_analysis/tasks/cvd/outputs"
RUN_ROOT = PROJECT_ROOT / "downstream_analysis/tasks" / NOTEBOOK_STEM
OUT_DIR = RUN_ROOT / "outputs"
CACHE_DIR = OUT_DIR / "cache"
FIG_DIR = OUT_DIR / "figures"
LOG_DIR = OUT_DIR / "logs"
for p in [RUN_ROOT, OUT_DIR, CACHE_DIR, FIG_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ID_COL = "participant_id"
FOOD_COL = "food_id"
GRAMS_COL = "weight_g"

FEATURE_SETS = [
    {"name": "basic_nutrimatch", "path": "outputs/enhanced_hpp/2.nutrimatch_based/hpp_feature_matrix_per_100g.csv", "feature_mode": "enriched", "label": "NutriMatch nutrients"},
    {"name": "denovo_enriched", "path": "outputs/enhanced_hpp/1.denovo/hpp_feature_matrix_per_100g.csv", "feature_mode": "enriched", "label": "De novo enriched nutrients"},
    {"name": "nutrimatch_enhanced", "path": "outputs/enhanced_hpp/3.nutrimatch_enhanced/hpp_feature_matrix_per_100g.csv", "feature_mode": "enriched", "label": "NutriMatch enhanced nutrients"},
    {"name": "denovo_cardiometabolic", "path": "outputs/downstream_features/denovo/cardiometabolic/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "De novo cardiometabolic"},
    {"name": "denovo_broad_diet_health", "path": "outputs/downstream_features/denovo/broad_diet_health/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "De novo broad diet-health"},
    {"name": "denovo_microbiome", "path": "outputs/downstream_features/denovo/microbiome/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "De novo microbiome-oriented"},
    {"name": "denovo_chemical_metabolomics", "path": "outputs/downstream_features/denovo/chemical_metabolomics/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "De novo chemical/metabolomics"},
    {"name": "denovo_mental_health", "path": "outputs/downstream_features/denovo/mental_health/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "De novo mental-health"},
    {"name": "nutrimatch_cardiometabolic", "path": "outputs/downstream_features/nutrimatch_based/cardiometabolic/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "NutriMatch cardiometabolic"},
    {"name": "nutrimatch_broad_diet_health", "path": "outputs/downstream_features/nutrimatch_based/broad_diet_health/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "NutriMatch broad diet-health"},
    {"name": "nutrimatch_microbiome", "path": "outputs/downstream_features/nutrimatch_based/microbiome/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "NutriMatch microbiome-oriented"},
    {"name": "nutrimatch_chemical_metabolomics", "path": "outputs/downstream_features/nutrimatch_based/chemical_metabolomics/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "NutriMatch chemical/metabolomics"},
    {"name": "nutrimatch_mental_health", "path": "outputs/downstream_features/nutrimatch_based/mental_health/hpp_downstream_feature_table.csv", "feature_mode": "kg", "label": "NutriMatch mental-health"},
    {"name": "denovo_food_card_embedding", "path": "outputs/food_card/denovo/embeddings/hpp_food_card_embeddings_full_biology_text_text_embedding_3_large.parquet", "feature_mode": "embedding", "label": "Food-card embedding"},
]

FEATURE_SETS = [fs for fs in FEATURE_SETS if (PROJECT_ROOT / fs["path"]).exists()]
print("Project root:", PROJECT_ROOT)
print("Run folder:", RUN_ROOT)
print("Output directory:", OUT_DIR)
print("Background runner:", PROJECT_ROOT / "downstream_analysis/Nutrimatch_manual/run_diet_data_enhancement_benchmark_I_background.py")
print("Feature sets found:", [fs["name"] for fs in FEATURE_SETS])


## Background Training Launcher

Run the next cell from the notebook/kernel. It starts the training script in a detached Python process, writes a PID file and log file, then returns immediately. This avoids the Jupyter `! ... &` limitation and uses the same Python environment as this kernel.


In [ ]:
# Console-only training command and status helper.
# The long benchmark should be launched from a terminal/console, not from the notebook browser session.
# In TRE/SageMaker, live logs are more reliable in /tmp than in the mounted project folder.

runner_path = PROJECT_ROOT / "downstream_analysis/Nutrimatch_manual/run_diet_data_enhancement_benchmark_I_background.py"
project_log_path = LOG_DIR / "background_training.log"
tmp_log_path = Path("/tmp/diet_data_enhancement_benchmark_I_background_training.log")
pid_path = LOG_DIR / "background_training.pid"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Run this from a TRE shell/terminal to create saved benchmark artifacts:")
print(f"cd {PROJECT_ROOT}")
print(f"mkdir -p {LOG_DIR}")
print(
    'DDE_RUN_TRAINING=1 PYTHONUNBUFFERED=1 '
    'PYTHONPATH="$PWD:${PYTHONPATH:-}" '
    f"nohup python -u {runner_path.relative_to(PROJECT_ROOT)} "
    f"> {tmp_log_path} 2>&1 &"
)
print(f"echo $! > {pid_path.relative_to(PROJECT_ROOT)}")
print(f"tail -f {tmp_log_path}")
print(f"cp {tmp_log_path} {project_log_path}")

print("\nCurrent saved-artifact status:")
print("Project log exists:", project_log_path.exists(), project_log_path)
print("Tmp live log exists:", tmp_log_path.exists(), tmp_log_path)
print("PID file exists:", pid_path.exists(), pid_path)
if pid_path.exists():
    print("PID:", pid_path.read_text(encoding="utf-8").strip())
combined_probe = OUT_DIR / "diet_data_enhancement_benchmark_I_all_metrics.csv"
print("Combined results exist:", combined_probe.exists(), combined_probe)
if combined_probe.exists():
    print("Combined results size:", combined_probe.stat().st_size)


## 2. TRE And PhenoLoader Helpers


In [ ]:
try:
    from downstream_analysis.data_handelling.pheno_loader_export import (
        dataframe_with_index_columns,
        load_table_from_loader,
        make_loader,
    )
except Exception as exc:
    print("Could not import local TRE helper module yet:", exc)
    dataframe_with_index_columns = None
    load_table_from_loader = None
    make_loader = None


def flatten_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out.index, pd.MultiIndex) or out.index.name is not None:
        out = out.reset_index()
    out = out.loc[:, ~out.columns.duplicated()].copy()
    return out


def read_any(path: str | Path, columns: list[str] | None = None) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {".parquet", ".pq"}:
        return flatten_index(pd.read_parquet(path, columns=columns))
    if suffix in {".feather", ".arrow"}:
        return flatten_index(pd.read_feather(path, columns=columns))
    if suffix in {".tsv", ".txt"}:
        return flatten_index(pd.read_csv(path, sep="\t", usecols=columns, low_memory=False))
    return flatten_index(pd.read_csv(path, usecols=columns, low_memory=False))


def find_participant_col(df: pd.DataFrame) -> str | None:
    for col in [ID_COL, "Participant_Study_ID", "research_stage_id", "user_id", "RegistrationCode", "sample_name", "sample_id"]:
        if col in df.columns:
            return col
    for col in df.columns:
        text = str(col).lower()
        if "participant" in text or "research_stage" in text or "sample" in text:
            return col
    return None


def normalize_ids(df: pd.DataFrame) -> pd.DataFrame:
    out = flatten_index(df)
    pid = find_participant_col(out)
    if pid is not None and pid != ID_COL:
        out = out.rename(columns={pid: ID_COL})
    if ID_COL in out.columns:
        out[ID_COL] = out[ID_COL].astype(str)
    return out


def try_load_pheno_table(dataset: str, table: str | None = None, errors: str = "warn") -> tuple[pd.DataFrame | None, Any | None]:
    table_name = table or dataset
    try:
        if make_loader is not None:
            loader = make_loader(dataset, age_sex_dataset=None, errors=errors)
        else:
            from pheno_utils import PhenoLoader
            loader = PhenoLoader(dataset, age_sex_dataset=None, errors=errors)
    except Exception as exc:
        print(f"Could not create PhenoLoader({dataset!r}): {exc}")
        return None, None

    df = None
    try:
        if load_table_from_loader is not None:
            df = load_table_from_loader(loader, dataset, table_name, required=False)
    except Exception:
        df = None

    if df is None:
        try:
            dfs = getattr(loader, "dfs", {})
            if table_name in dfs:
                df = flatten_index(dfs[table_name])
            else:
                candidate = loader[table_name]
                if isinstance(candidate, pd.Series):
                    candidate = candidate.to_frame()
                if isinstance(candidate, pd.DataFrame):
                    df = flatten_index(candidate)
        except Exception:
            df = None

    if df is None:
        print(f"Could not load {dataset}/{table_name}; available pl.dfs tables: {sorted(getattr(loader, 'dfs', {}).keys())}")
        return None, loader
    return normalize_ids(df), loader


def dataset_dir_from_loader(loader, dataset: str) -> Path | None:
    try:
        from pheno_utils.config import DATASETS_PATH
        folder = getattr(loader, "dataset", dataset)
        return Path(DATASETS_PATH) / str(folder)
    except Exception:
        return None


def safe_numeric_columns(df: pd.DataFrame, exclude: set[str] | None = None) -> list[str]:
    exclude = exclude or set()
    bad = re.compile(r"participant|cohort|research_stage|collection|timestamp|date|timezone|path|file|parquet|arrow|tsv|fastq|html|zip|tar|qc|sample|array_index", re.I)
    return [c for c in df.select_dtypes(include=[np.number, bool]).columns if c not in exclude and not bad.search(str(c))]


## 3. Build Or Reuse Diet X Matrices

This is the expensive part if the cache is missing. With the previous CVD run completed, these files should already exist and will be reused.


In [ ]:
DIET_SLIM_CSV = TRE_INPUTS / "diet_participant_food_slim.csv"
DIET_EVENTS_CSV = TRE_INPUTS / "diet_logging_events.csv"
DIET_EVENTS_PARQUET = TRE_INPUTS / "diet_logging_events.parquet"


def ensure_diet_slim() -> Path:
    if DIET_SLIM_CSV.exists() and not FORCE_REBUILD_X:
        print("Reusing slim diet table:", DIET_SLIM_CSV)
        return DIET_SLIM_CSV

    TRE_INPUTS.mkdir(parents=True, exist_ok=True)
    if DIET_EVENTS_CSV.exists():
        source = DIET_EVENTS_CSV
        reader = pd.read_csv(source, usecols=[ID_COL, FOOD_COL, GRAMS_COL], chunksize=500_000, low_memory=False)
    elif DIET_EVENTS_PARQUET.exists():
        source = DIET_EVENTS_PARQUET
        df = pd.read_parquet(source, columns=[ID_COL, FOOD_COL, GRAMS_COL])
        reader = [df]
    else:
        print("diet_logging_events not cached; loading from PhenoLoader...")
        diet, _ = try_load_pheno_table("diet_logging", "diet_logging_events")
        if diet is None:
            raise FileNotFoundError("Could not load diet_logging_events from PhenoLoader.")
        keep = [c for c in [ID_COL, FOOD_COL, GRAMS_COL] if c in diet.columns]
        if len(keep) < 3:
            raise ValueError(f"Diet table missing required columns. Found: {keep}")
        diet[keep].to_csv(DIET_EVENTS_CSV, index=False)
        source = DIET_EVENTS_CSV
        reader = pd.read_csv(source, usecols=[ID_COL, FOOD_COL, GRAMS_COL], chunksize=500_000, low_memory=False)

    print("Building slim diet table from:", source)
    grouped_chunks = []
    for chunk in reader:
        chunk = normalize_ids(chunk)
        chunk[FOOD_COL] = chunk[FOOD_COL].astype(str)
        chunk[GRAMS_COL] = pd.to_numeric(chunk[GRAMS_COL], errors="coerce").fillna(0.0).clip(lower=0)
        grouped_chunks.append(chunk.groupby([ID_COL, FOOD_COL], as_index=False)[GRAMS_COL].sum())
    diet_slim = pd.concat(grouped_chunks, ignore_index=True).groupby([ID_COL, FOOD_COL], as_index=False)[GRAMS_COL].sum()
    diet_slim.to_csv(DIET_SLIM_CSV, index=False)
    del grouped_chunks, diet_slim
    gc.collect()
    print("Wrote slim diet table:", DIET_SLIM_CSV)
    return DIET_SLIM_CSV


def choose_ref_food_col(ref: pd.DataFrame) -> str:
    for col in ["hpp_food_id", "food_id", "FoodID", "food_code"]:
        if col in ref.columns:
            return col
    raise ValueError("Could not find hpp_food_id or food_id in feature table.")


def feature_columns(ref: pd.DataFrame, ref_food_col: str, feature_mode: str) -> list[str]:
    if feature_mode == "embedding":
        cols = [c for c in ref.columns if str(c).startswith("embedding_")]
    else:
        exclude = {ref_food_col, "food_id", "hpp_food_id", "food_name", "short_food_name", "product_name", "description", "category"}
        cols = [c for c in ref.columns if c not in exclude and pd.api.types.is_numeric_dtype(ref[c])]
    return list(dict.fromkeys(cols))


def build_participant_x(fs: dict, batch_size: int = 100) -> Path:
    fs_name = fs["name"]
    existing = CVD_OUTPUTS / fs_name / f"X_{fs_name}_participant.csv"
    if existing.exists() and not FORCE_REBUILD_X:
        print("Reusing X:", existing)
        return existing

    diet_slim_csv = ensure_diet_slim()
    print("Building X for", fs_name)
    ref = read_any(PROJECT_ROOT / fs["path"])
    ref_food_col = choose_ref_food_col(ref)
    cols = feature_columns(ref, ref_food_col, fs["feature_mode"])
    if not cols:
        raise ValueError(f"No feature columns found for {fs_name}")
    ref = ref[[ref_food_col] + cols].copy()
    ref["_food_join_id"] = ref[ref_food_col].astype(str)

    diet = pd.read_csv(diet_slim_csv, low_memory=False)
    diet[ID_COL] = diet[ID_COL].astype(str)
    diet["_food_join_id"] = diet[FOOD_COL].astype(str)
    diet[GRAMS_COL] = pd.to_numeric(diet[GRAMS_COL], errors="coerce").fillna(0.0).clip(lower=0)
    total_grams = diet.groupby(ID_COL)[GRAMS_COL].sum().replace(0, np.nan)

    parts = []
    for start in range(0, len(cols), batch_size):
        batch = cols[start:start + batch_size]
        merged = diet[[ID_COL, "_food_join_id", GRAMS_COL]].merge(ref[["_food_join_id"] + batch], on="_food_join_id", how="left")
        values = merged[batch].apply(pd.to_numeric, errors="coerce").fillna(0.0)
        if fs["feature_mode"] == "enriched":
            scaled = values.mul(merged[GRAMS_COL].to_numpy() / 100.0, axis=0)
            prefix = "enriched_"
        elif fs["feature_mode"] in {"kg", "embedding"}:
            scaled = values.mul(merged[GRAMS_COL].to_numpy(), axis=0)
            prefix = "kg_" if fs["feature_mode"] == "kg" else "food_card_"
        else:
            raise ValueError(fs["feature_mode"])
        scaled[ID_COL] = merged[ID_COL].values
        agg = scaled.groupby(ID_COL).sum(numeric_only=True)
        if fs["feature_mode"] in {"kg", "embedding"}:
            agg = agg.div(total_grams, axis=0).fillna(0.0)
        agg.columns = [prefix + str(c) for c in agg.columns]
        parts.append(agg)
        del merged, values, scaled, agg
        gc.collect()

    x = pd.concat(parts, axis=1).reset_index()
    fs_dir = CVD_OUTPUTS / fs_name
    fs_dir.mkdir(parents=True, exist_ok=True)
    out = fs_dir / f"X_{fs_name}_participant.csv"
    x.to_csv(out, index=False)
    print("Wrote X:", out)
    return out


def build_or_reuse_all_x() -> dict[str, Path]:
    ensure_diet_slim()
    return {fs["name"]: build_participant_x(fs) for fs in FEATURE_SETS}

x_paths = build_or_reuse_all_x()
x_paths


## 4. Model Arms


In [ ]:
PAPER_BASIC_NUTRIENT_PATTERNS = [
    r"energy|calor",
    r"protein",
    r"total lipid|lipid|fat$|\bfat\b",
    r"carbohydrate|carb",
    r"sodium|\bna\b",
    r"fiber|fibre",
    r"alcohol",
    r"water",
]


def clean_feature_name(name: object) -> str:
    text = str(name)
    for prefix in ["enriched_", "kg_", "food_card_"]:
        if text.startswith(prefix):
            text = text[len(prefix):]
    text = re.sub(r"[_\-]+", " ", text)
    return re.sub(r"\s+", " ", text).strip().lower()


def load_x(feature_set: str) -> pd.DataFrame:
    x = pd.read_csv(x_paths[feature_set], low_memory=False)
    x = normalize_ids(x)
    if "time_window" in x.columns:
        x = x.drop(columns=["time_window"])
    return x


def select_base_nutrient_cols(basic_x: pd.DataFrame) -> list[str]:
    cols = [c for c in basic_x.columns if c != ID_COL and pd.api.types.is_numeric_dtype(basic_x[c])]
    selected = []
    for col in cols:
        cleaned = clean_feature_name(col)
        if any(re.search(pat, cleaned, re.I) for pat in PAPER_BASIC_NUTRIENT_PATTERNS):
            selected.append(col)
    return list(dict.fromkeys(selected))


def build_covariates() -> pd.DataFrame:
    cache = CACHE_DIR / "covariates_age_sex.csv"
    if cache.exists() and not FORCE_REBUILD_TARGETS:
        cov = pd.read_csv(cache, low_memory=False)
        return normalize_ids(cov)

    parts = []
    for dataset in ["anthropometrics", "body_composition", "blood_tests", "cgm", "gut_microbiome", "oral_microbiome", "vascular_health", "nightingale_metabolomics", "population"]:
        for table in ["age_sex", "population"] if dataset == "population" else ["age_sex"]:
            df, _ = try_load_pheno_table(dataset, table)
            if df is None or ID_COL not in df.columns:
                continue
            matches = [c for c in df.columns if re.search(r"^age$|age_at|sex$|gender$|year_of_birth", str(c), re.I)]
            if not matches:
                continue
            part = df[[ID_COL] + matches].copy()
            rename = {}
            for c in matches:
                lc = str(c).lower()
                if "sex" in lc or "gender" in lc:
                    rename[c] = "sex"
                elif "year_of_birth" in lc:
                    rename[c] = "year_of_birth"
                elif "age" in lc:
                    rename[c] = "age"
            part = part.rename(columns=rename)
            keep = [ID_COL] + [c for c in ["age", "sex", "year_of_birth"] if c in part.columns]
            part = part[keep].copy()
            if "age" in part.columns:
                part["age"] = pd.to_numeric(part["age"], errors="coerce")
            if "year_of_birth" in part.columns and "age" not in part.columns:
                part["year_of_birth"] = pd.to_numeric(part["year_of_birth"], errors="coerce")
                part["age"] = 2022 - part["year_of_birth"]
            if "year_of_birth" in part.columns:
                part = part.drop(columns=["year_of_birth"])
            part = part.groupby(ID_COL, as_index=False).first()
            parts.append(part)

    if not parts:
        cov = pd.DataFrame({ID_COL: pd.Series(dtype=str)})
    else:
        cov = parts[0]
        for part in parts[1:]:
            for col in [c for c in part.columns if c != ID_COL]:
                if col not in cov.columns:
                    cov = cov.merge(part[[ID_COL, col]], on=ID_COL, how="outer")
                else:
                    add = part[[ID_COL, col]].rename(columns={col: f"{col}_new"})
                    cov = cov.merge(add, on=ID_COL, how="outer")
                    cov[col] = cov[col].combine_first(cov[f"{col}_new"])
                    cov = cov.drop(columns=[f"{col}_new"])
    cov.to_csv(cache, index=False)
    return cov


covariates = build_covariates()
basic_x = load_x("basic_nutrimatch")
base_nutrient_cols = select_base_nutrient_cols(basic_x)
covariate_cols = [c for c in ["age", "sex"] if c in covariates.columns and covariates[c].notna().any()]

print("Covariates:", covariate_cols)
print("Base nutrient columns:", base_nutrient_cols)
if not base_nutrient_cols:
    raise ValueError("No base nutrient columns found in basic_nutrimatch. Check PAPER_BASIC_NUTRIENT_PATTERNS.")


def with_covariates(x: pd.DataFrame) -> pd.DataFrame:
    x = normalize_ids(x)
    if covariate_cols:
        return x.merge(covariates[[ID_COL] + covariate_cols], on=ID_COL, how="left")
    return x


MODEL_ARMS = [
    {"arm": "age_sex", "feature_set": "covariates", "label": "Age + sex", "x": covariates[[ID_COL] + covariate_cols].copy()},
    {"arm": "base_nutrients", "feature_set": "basic_nutrimatch", "label": "Age + sex + base nutrients", "x": with_covariates(basic_x[[ID_COL] + base_nutrient_cols].copy())},
    {"arm": "nutrimatch", "feature_set": "basic_nutrimatch", "label": "Age + sex + NutriMatch", "x": with_covariates(basic_x.copy())},
]
for fs in FEATURE_SETS:
    if fs["name"] == "basic_nutrimatch":
        continue
    MODEL_ARMS.append({"arm": fs["name"], "feature_set": fs["name"], "label": "Age + sex + " + fs["label"], "x": with_covariates(load_x(fs["name"]))})

print("Model arms:")
for arm in MODEL_ARMS:
    print(f"{arm['arm']:35s}", arm["x"].shape)


## 5. Target Loading Helpers

Direct phenotype targets are read from PhenoLoader tables. Microbiome targets first use the main table, then try linked abundance/pathway tables if local paths are exposed in the TRE mount. Prepared target matrices are cached per benchmark section.


In [ ]:
def cache_paths_for_section(section_id: str) -> tuple[Path, Path]:
    return CACHE_DIR / f"{section_id}_targets.csv", CACHE_DIR / f"{section_id}_target_catalog.csv"


def target_id_from(section_id: str, dataset: str, table: str, column: str) -> str:
    raw = f"{section_id}__{dataset}__{table}__{column}"
    return re.sub(r"[^A-Za-z0-9_]+", "_", raw).strip("_")[:180]


def collapse_target(frame: pd.DataFrame, column: str, target_id: str, agg: str = "mean") -> pd.DataFrame:
    frame = normalize_ids(frame)
    if ID_COL not in frame.columns or column not in frame.columns:
        return pd.DataFrame(columns=[ID_COL, target_id])
    date_cols = [c for c in ["collection_date", "collection_timestamp", "date", "timestamp"] if c in frame.columns]
    part = frame[[ID_COL, column] + date_cols].copy().rename(columns={column: target_id})
    part[target_id] = pd.to_numeric(part[target_id], errors="coerce")
    part = part.dropna(subset=[target_id])
    if part.empty:
        return pd.DataFrame(columns=[ID_COL, target_id])
    if agg == "first" and date_cols:
        date_col = date_cols[0]
        part[date_col] = pd.to_datetime(part[date_col], errors="coerce")
        part = part.sort_values([ID_COL, date_col]).groupby(ID_COL, as_index=False).first()
        return part[[ID_COL, target_id]]
    return part.groupby(ID_COL, as_index=False)[target_id].mean()


def select_target_columns(frame: pd.DataFrame, include_regex: str | None = None, exclude_regex: str | None = None, max_targets: int = MAX_TARGETS_PER_SECTION) -> list[str]:
    exclude = {ID_COL}
    numeric = safe_numeric_columns(frame, exclude=exclude)
    if include_regex:
        rx = re.compile(include_regex, re.I)
        numeric = [c for c in numeric if rx.search(str(c))]
    if exclude_regex:
        ex = re.compile(exclude_regex, re.I)
        numeric = [c for c in numeric if not ex.search(str(c))]
    scored = []
    for col in numeric:
        x = pd.to_numeric(frame[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        n = int(x.notna().sum())
        if n < MIN_N_PER_TARGET or x.nunique(dropna=True) < 5:
            continue
        variance = float(np.nanvar(x))
        nonzero = float((x.fillna(0) != 0).mean())
        scored.append((n, variance, nonzero, col))
    scored = sorted(scored, reverse=True)
    return [col for *_score, col in scored[:max_targets]]


def clean_path_value(value: object) -> str | None:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "null"} or text.startswith("s3://"):
        return None
    return text


def candidate_bulk_paths(raw: object, dataset_dir: Path | None) -> list[Path]:
    text = clean_path_value(raw)
    if not text:
        return []
    cleaned = text.lstrip("./")
    paths = [Path(text)]
    if dataset_dir is not None:
        paths.extend([
            dataset_dir / text,
            dataset_dir / cleaned,
            dataset_dir.parent / text,
            dataset_dir.parent / cleaned,
        ])
    out, seen = [], set()
    for path in paths:
        key = str(path)
        if key not in seen:
            seen.add(key)
            out.append(path)
    return out


def normalize_abundance_table(df: pd.DataFrame, *, prefix: str, participant_lookup: dict[str, str] | None = None) -> pd.DataFrame:
    out = normalize_ids(df)
    pid = find_participant_col(out)
    if pid is not None and pid != ID_COL:
        out = out.rename(columns={pid: ID_COL})

    name_cols = [c for c in out.columns if re.search(r"taxon|clade|species|genus|family|pathway|feature|name", str(c), re.I)]
    value_cols = [c for c in out.select_dtypes(include=[np.number]).columns if re.search(r"abundance|coverage|relative|value|count|rpm|cpm|percent", str(c), re.I)]

    if ID_COL in out.columns and name_cols and value_cols and len(value_cols) <= 4:
        wide = out.pivot_table(index=ID_COL, columns=name_cols[0], values=value_cols[0], aggfunc="mean", fill_value=0)
        wide.columns = [f"{prefix}__{re.sub(r'[^A-Za-z0-9]+', '_', str(c)).strip('_').lower()[:120]}" for c in wide.columns]
        return wide.reset_index()

    features = safe_numeric_columns(out, exclude={ID_COL})
    if ID_COL in out.columns and features:
        wide = out[[ID_COL] + features].copy()
        wide = wide.groupby(ID_COL, as_index=False).mean(numeric_only=True)
        wide = wide.rename(columns={c: f"{prefix}__{re.sub(r'[^A-Za-z0-9]+', '_', str(c)).strip('_').lower()[:120]}" for c in features})
        return wide

    if name_cols:
        numeric = out.set_index(name_cols[0]).select_dtypes(include=[np.number])
        if numeric.shape[1] > 1:
            wide = numeric.T.reset_index().rename(columns={"index": ID_COL})
            if participant_lookup:
                ids = wide[ID_COL].astype(str)
                mapped = ids.map(participant_lookup).fillna(ids.map(lambda x: participant_lookup.get(Path(x).name, np.nan))).fillna(ids)
                wide[ID_COL] = mapped.astype(str)
            wide.columns = [ID_COL if c == ID_COL else f"{prefix}__{re.sub(r'[^A-Za-z0-9]+', '_', str(c)).strip('_').lower()[:120]}" for c in wide.columns]
            features = [c for c in wide.columns if c != ID_COL]
            return wide.groupby(ID_COL, as_index=False)[features].mean()

    raise ValueError(f"Could not normalize abundance table with shape={out.shape}")


def participant_lookup_from_primary(primary: pd.DataFrame) -> dict[str, str]:
    primary = normalize_ids(primary)
    if ID_COL not in primary.columns:
        return {}
    key_cols = [c for c in primary.columns if re.search(r"sample|fastq|path|file|parquet|arrow|tsv|wgs|dna", str(c), re.I)]
    lookup = {}
    for _, row in primary[[ID_COL] + key_cols].dropna(subset=[ID_COL]).iterrows():
        pid = str(row[ID_COL])
        for col in key_cols:
            val = clean_path_value(row.get(col))
            if not val:
                continue
            lookup[val] = pid
            lookup[Path(val).name] = pid
            lookup[Path(val).stem] = pid
    return lookup


def alpha_diversity(table: pd.DataFrame, prefix: str) -> pd.DataFrame:
    table = normalize_ids(table)
    features = [c for c in table.select_dtypes(include=[np.number]).columns if c != ID_COL]
    if not features:
        return table[[ID_COL]].copy()
    x = table[features].apply(pd.to_numeric, errors="coerce").clip(lower=0).fillna(0).to_numpy(dtype=float)
    sums = x.sum(axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.divide(x, sums[:, None], out=np.zeros_like(x), where=sums[:, None] > 0)
        logp = np.where(p > 0, np.log(p), 0.0)
    return pd.DataFrame({
        ID_COL: table[ID_COL].astype(str).to_numpy(),
        f"{prefix}_shannon": -(p * logp).sum(axis=1),
        f"{prefix}_simpson": 1.0 - (p ** 2).sum(axis=1),
        f"{prefix}_richness": (x > 0).sum(axis=1),
        f"{prefix}_total_abundance": sums,
    })


def load_microbiome_targets(section_id: str, dataset: str, primary_table: str, bulk_patterns: list[tuple[str, str]], include_regex: str, max_targets: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    target_cache, catalog_cache = cache_paths_for_section(section_id)
    if target_cache.exists() and catalog_cache.exists() and not FORCE_REBUILD_TARGETS:
        return pd.read_csv(target_cache, low_memory=False), pd.read_csv(catalog_cache, low_memory=False)

    primary, loader = try_load_pheno_table(dataset, primary_table)
    if primary is None:
        raise ValueError(f"Could not load {dataset}/{primary_table}")
    dataset_dir = dataset_dir_from_loader(loader, dataset)
    lookup = participant_lookup_from_primary(primary)

    matrices = []
    catalogs = []

    primary_cols = select_target_columns(primary, include_regex=include_regex, max_targets=max(6, max_targets // 3))
    primary_part = pd.DataFrame({ID_COL: primary[ID_COL].astype(str).unique()}) if ID_COL in primary.columns else pd.DataFrame(columns=[ID_COL])
    for col in primary_cols:
        tid = target_id_from(section_id, dataset, primary_table, col)
        part = collapse_target(primary, col, tid)
        if not part.empty:
            primary_part = primary_part.merge(part, on=ID_COL, how="outer")
            catalogs.append({"section": section_id, "target_id": tid, "target_label": col, "target_group": "main_table", "dataset": dataset, "table": primary_table, "column": col, "source": "primary"})
    if len(primary_part.columns) > 1:
        matrices.append(primary_part)

    path_cols = [c for c in primary.columns if re.search(r"parquet|arrow|tsv|metaphlan|humann|urs", str(c), re.I)]
    for label, pattern in bulk_patterns:
        rx = re.compile(pattern, re.I)
        matched_path_cols = [c for c in path_cols if rx.search(str(c))]
        for path_col in matched_path_cols[:3]:
            loaded = None
            loaded_path = None
            for raw in primary[path_col].dropna().astype(str).unique()[:30]:
                for candidate in candidate_bulk_paths(raw, dataset_dir):
                    if candidate.exists():
                        loaded = read_any(candidate)
                        loaded_path = candidate
                        break
                if loaded is not None:
                    break
            if loaded is None:
                continue
            try:
                matrix = normalize_abundance_table(loaded, prefix=label, participant_lookup=lookup)
                matrix = normalize_ids(matrix)
                matrices.append(alpha_diversity(matrix, label))
                cols = select_target_columns(matrix, include_regex=None, max_targets=max_targets)
                selected = matrix[[ID_COL] + cols].copy()
                matrices.append(selected)
                for col in cols:
                    catalogs.append({"section": section_id, "target_id": col, "target_label": col, "target_group": label, "dataset": dataset, "table": path_col, "column": col, "source": str(loaded_path)})
                for col in [c for c in matrices[-2].columns if c != ID_COL]:
                    catalogs.append({"section": section_id, "target_id": col, "target_label": col, "target_group": f"{label}_diversity", "dataset": dataset, "table": path_col, "column": col, "source": str(loaded_path)})
            except Exception as exc:
                print(f"Could not normalize {dataset}/{path_col}: {exc}")

    if not matrices:
        raise ValueError(f"No usable targets found for {section_id}")
    targets = matrices[0]
    for m in matrices[1:]:
        targets = targets.merge(m, on=ID_COL, how="outer")
    targets = targets.loc[:, ~targets.columns.duplicated()].copy()

    catalog = pd.DataFrame(catalogs).drop_duplicates("target_id")
    keep = [ID_COL] + [c for c in catalog["target_id"].tolist() if c in targets.columns]
    targets = targets[keep]
    targets.to_csv(target_cache, index=False)
    catalog.to_csv(catalog_cache, index=False)
    return targets, catalog


def load_direct_targets(section_id: str, table_specs: list[dict], max_targets: int = MAX_TARGETS_PER_SECTION) -> tuple[pd.DataFrame, pd.DataFrame]:
    target_cache, catalog_cache = cache_paths_for_section(section_id)
    if target_cache.exists() and catalog_cache.exists() and not FORCE_REBUILD_TARGETS:
        return pd.read_csv(target_cache, low_memory=False), pd.read_csv(catalog_cache, low_memory=False)

    target_parts = []
    catalogs = []
    for spec in table_specs:
        dataset = spec["dataset"]
        table = spec.get("table", dataset)
        frame, _loader = try_load_pheno_table(dataset, table)
        if frame is None or ID_COL not in frame.columns:
            continue
        cols = spec.get("columns")
        if cols is None:
            cols = select_target_columns(frame, include_regex=spec.get("include_regex"), exclude_regex=spec.get("exclude_regex"), max_targets=spec.get("max_targets", max_targets))
        else:
            cols = [c for c in cols if c in frame.columns]
        for col in cols:
            tid = target_id_from(section_id, dataset, table, col)
            part = collapse_target(frame, col, tid, agg=spec.get("agg", "mean"))
            if part.empty or part[tid].notna().sum() < MIN_N_PER_TARGET:
                continue
            target_parts.append(part)
            catalogs.append({
                "section": section_id,
                "target_id": tid,
                "target_label": spec.get("label_map", {}).get(col, col),
                "target_group": spec.get("target_group", dataset),
                "dataset": dataset,
                "table": table,
                "column": col,
                "source": f"PhenoLoader({dataset!r})/{table}",
            })
    if not target_parts:
        raise ValueError(f"No direct targets found for {section_id}")
    targets = target_parts[0]
    for part in target_parts[1:]:
        targets = targets.merge(part, on=ID_COL, how="outer")
    catalog = pd.DataFrame(catalogs).drop_duplicates("target_id")
    targets.to_csv(target_cache, index=False)
    catalog.to_csv(catalog_cache, index=False)
    return targets, catalog


## 6. Classification Versions Of Continuous Targets

Clinical thresholds are used when they are obvious; otherwise the notebook creates quantile-based high/low and quartile labels.


In [ ]:
def clinical_rule(label: str):
    text = str(label).lower()
    rules = [
        (r"hba1c|hemoglobin.*a1c|ea1c|gmi", ">=", 5.7, "clinical_high"),
        (r"fasting.*glucose|bt__glucose|glucose_float|cgm_mean|average.*glucose|mean.*glucose", ">=", 100.0, "clinical_high"),
        (r"bmi|body_mass_index", ">=", 25.0, "clinical_high"),
        (r"triglyceride", ">=", 150.0, "clinical_high"),
        (r"ldl", ">=", 130.0, "clinical_high"),
        (r"total.*cholesterol|total_c", ">=", 200.0, "clinical_high"),
        (r"abi", "<=", 0.90, "clinical_low"),
        (r"in_range_70_180", "<", 70.0, "clinical_low"),
        (r"above_180", ">=", 10.0, "clinical_high"),
        (r"folate|folic", "<", 4.0, "clinical_low"),
    ]
    for pattern, op, threshold, name in rules:
        if re.search(pattern, text, re.I):
            return op, threshold, name
    return None


def add_classification_targets(targets: pd.DataFrame, catalog: pd.DataFrame, max_new: int = MAX_CLASSIFICATION_TARGETS_PER_SECTION) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not RUN_CLASSIFICATION_DERIVATIVES:
        return targets, catalog
    out = targets.copy()
    new_rows = []
    added = 0
    for _, row in catalog.iterrows():
        if added >= max_new:
            break
        col = row["target_id"]
        if col not in out.columns:
            continue
        y = pd.to_numeric(out[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        if y.notna().sum() < MIN_N_PER_TARGET or y.nunique(dropna=True) < 8:
            continue

        rule = clinical_rule(row.get("target_label", col))
        if rule is not None:
            op, threshold, rule_name = rule
            new_col = f"{col}__{rule_name}"
            if op == ">=":
                out[new_col] = (y >= threshold).where(y.notna()).astype("float")
            elif op == "<=":
                out[new_col] = (y <= threshold).where(y.notna()).astype("float")
            else:
                out[new_col] = (y < threshold).where(y.notna()).astype("float")
            if out[new_col].value_counts(dropna=True).min() >= MIN_CLASS_COUNT:
                new_rows.append({**row.to_dict(), "target_id": new_col, "target_label": f"{row.get('target_label', col)} [{rule_name} {op} {threshold}]", "target_group": row.get("target_group", "") + "/classification", "derived_from": col, "threshold_rule": f"{op} {threshold}"})
                added += 1

        if added >= max_new:
            break
        try:
            q = pd.qcut(y, q=4, labels=False, duplicates="drop")
            if q.nunique(dropna=True) >= 3 and q.value_counts(dropna=True).min() >= MIN_CLASS_COUNT:
                q_col = f"{col}__quartile_class"
                out[q_col] = q.astype("float")
                new_rows.append({**row.to_dict(), "target_id": q_col, "target_label": f"{row.get('target_label', col)} [quartiles]", "target_group": row.get("target_group", "") + "/classification", "derived_from": col, "threshold_rule": "quartiles"})
                added += 1
        except Exception:
            pass
    if new_rows:
        catalog = pd.concat([catalog, pd.DataFrame(new_rows)], ignore_index=True)
    return out, catalog


def infer_target_task(y: pd.Series, target_id: str, catalog_row: pd.Series) -> tuple[str, int | None]:
    y2 = y.dropna()
    if target_id.endswith("__clinical_high") or target_id.endswith("__clinical_low") or target_id.endswith("__quartile_class"):
        return "classification", int(y2.nunique())
    if y2.nunique() <= 2:
        return "classification", int(y2.nunique())
    return "regression", None


## 7. Evaluation And Plotting Helpers


In [ ]:
def reduce_features_unsupervised(X: pd.DataFrame, max_features: int = MAX_FEATURES_PER_ARM) -> pd.DataFrame:
    X = X.copy()
    numeric = X.select_dtypes(include=[np.number, bool]).columns.tolist()
    other = [c for c in X.columns if c not in numeric]
    if len(numeric) <= max_features:
        return X
    variances = X[numeric].apply(pd.to_numeric, errors="coerce").var(axis=0, skipna=True).sort_values(ascending=False)
    keep_numeric = variances.head(max_features).index.tolist()
    return X[keep_numeric + other]


def model_for(task_type: str, model_name: str):
    if task_type == "regression":
        if model_name == "ridge":
            return Ridge(alpha=10.0, random_state=RANDOM_STATE)
        if model_name == "random_forest":
            return RandomForestRegressor(n_estimators=RF_N_ESTIMATORS, min_samples_leaf=5, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1)
    else:
        if model_name == "ridge":
            return RidgeClassifier(alpha=10.0)
        if model_name == "random_forest":
            return RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, min_samples_leaf=5, max_features="sqrt", class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
    raise ValueError((task_type, model_name))


def pipeline_for(X: pd.DataFrame, task_type: str, model_name: str) -> Pipeline:
    numeric = X.select_dtypes(include=[np.number, bool]).columns.tolist()
    cats = [c for c in X.columns if c not in numeric]
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric),
            ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cats),
        ],
        remainder="drop",
    )
    return Pipeline([("pre", pre), ("model", model_for(task_type, model_name))])


def usable_cv(y: pd.Series, task_type: str, n_splits: int = N_SPLITS):
    if task_type == "classification":
        counts = y.value_counts(dropna=True)
        if counts.empty or counts.min() < 2:
            return None
        k = min(n_splits, int(counts.min()))
        if k < 2:
            return None
        return StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    k = min(n_splits, max(2, int(y.notna().sum() // 30)))
    return KFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)


def classification_scores(estimator: Pipeline, X: pd.DataFrame, y: pd.Series, cv):
    try:
        pred = cross_val_predict(clone(estimator), X, y, cv=cv, method="predict")
    except Exception as exc:
        print("classification predict failed:", exc)
        return None, None
    score = None
    try:
        score = cross_val_predict(clone(estimator), X, y, cv=cv, method="predict_proba")
    except Exception:
        try:
            score = cross_val_predict(clone(estimator), X, y, cv=cv, method="decision_function")
        except Exception:
            score = pred
    return pred, score


def evaluate_predictions(y_true: pd.Series, pred, score, task_type: str, class_count: int | None) -> dict[str, float]:
    if task_type == "regression":
        rmse = float(np.sqrt(mean_squared_error(y_true, pred)))
        try:
            pr = float(pearsonr(y_true, pred).statistic)
        except Exception:
            pr = np.nan
        return {"r2": float(r2_score(y_true, pred)), "rmse": rmse, "pearson_r": pr, "accuracy": np.nan, "auroc": np.nan, "auprc": np.nan, "f1_macro": np.nan, "balanced_accuracy": np.nan}

    metrics = {
        "r2": np.nan,
        "rmse": np.nan,
        "pearson_r": np.nan,
        "accuracy": float(accuracy_score(y_true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "f1_macro": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "auroc": np.nan,
        "auprc": np.nan,
    }
    try:
        if class_count == 2:
            labels = sorted(pd.Series(y_true).dropna().unique())
            positive = labels[-1]
            if isinstance(score, np.ndarray) and score.ndim == 2:
                score_1 = score[:, -1]
            else:
                score_1 = np.asarray(score)
            metrics["auroc"] = float(roc_auc_score(y_true, score_1))
            metrics["auprc"] = float(average_precision_score((y_true == positive).astype(int), score_1))
        elif class_count and class_count > 2:
            lb = LabelBinarizer()
            y_bin = lb.fit_transform(y_true)
            if isinstance(score, np.ndarray) and score.ndim == 2 and score.shape[1] == y_bin.shape[1]:
                metrics["auroc"] = float(roc_auc_score(y_bin, score, average="macro"))
                metrics["auprc"] = float(average_precision_score(y_bin, score, average="macro"))
    except Exception:
        pass
    return metrics


def make_oof_predictions(X: pd.DataFrame, y: pd.Series, task_type: str, model_name: str):
    valid = y.notna()
    X = reduce_features_unsupervised(X.loc[valid].copy())
    y = y.loc[valid].copy()
    if len(y) < MIN_N_PER_TARGET:
        return None
    cv = usable_cv(y, task_type)
    if cv is None:
        return None
    estimator = pipeline_for(X, task_type, model_name)
    if task_type == "regression":
        pred = cross_val_predict(estimator, X, y, cv=cv, method="predict")
        return X, y, pred, pred
    pred, score = classification_scores(estimator, X, y, cv)
    if pred is None:
        return None
    return X, y, pred, score


def evaluate_one_arm(arm: dict, y_table: pd.DataFrame, target_id: str, task_type: str, class_count: int | None, model_name: str) -> dict | None:
    merged = arm["x"].merge(y_table[[ID_COL, target_id]], on=ID_COL, how="inner")
    y_raw = pd.to_numeric(merged[target_id], errors="coerce")
    valid = y_raw.notna()
    if valid.sum() < MIN_N_PER_TARGET:
        return None
    merged = merged.loc[valid].copy()
    y = y_raw.loc[valid].copy()
    if task_type == "classification":
        y = y.astype("Int64").astype(str)
        if y.value_counts().min() < MIN_CLASS_COUNT:
            return None
    X = merged.drop(columns=[ID_COL, target_id])
    oof = make_oof_predictions(X, y, task_type, model_name)
    if oof is None:
        return None
    X2, y2, pred, score = oof
    metrics = evaluate_predictions(y2, pred, score, task_type, class_count)
    task_label = f"classification ({class_count} classes)" if task_type == "classification" else "regression"
    return {"arm": arm["arm"], "feature_set": arm["feature_set"], "label": arm["label"], "model": model_name, "task": task_label, "task_type": task_type, "class_count": class_count, "n": int(len(y2)), "feature_count": int(X2.shape[1]), **metrics}


def run_benchmark_section(section_id: str, targets: pd.DataFrame, catalog: pd.DataFrame) -> pd.DataFrame:
    targets = normalize_ids(targets)
    targets, catalog = add_classification_targets(targets, catalog)
    section_dir = OUT_DIR / section_id
    section_dir.mkdir(parents=True, exist_ok=True)
    targets.to_csv(section_dir / f"{section_id}_targets_with_classifications.csv", index=False)
    catalog.to_csv(section_dir / f"{section_id}_target_catalog_with_classifications.csv", index=False)

    rows = []
    for _, target_meta in catalog.iterrows():
        target_id = target_meta["target_id"]
        if target_id not in targets.columns:
            continue
        y = pd.to_numeric(targets[target_id], errors="coerce")
        task_type, class_count = infer_target_task(y, target_id, target_meta)
        print(f"[{section_id}] {target_meta.get('target_label', target_id)} | {task_type}")
        for arm in MODEL_ARMS:
            for model_name in ["ridge", "random_forest"]:
                row = evaluate_one_arm(arm, targets, target_id, task_type, class_count, model_name)
                if row is None:
                    continue
                rows.append({
                    "section": section_id,
                    "target_id": target_id,
                    "target_label": target_meta.get("target_label", target_id),
                    "target_group": target_meta.get("target_group", section_id),
                    "dataset": target_meta.get("dataset", ""),
                    "table": target_meta.get("table", ""),
                    "column": target_meta.get("column", ""),
                    **row,
                })
    results = pd.DataFrame(rows)
    out = section_dir / f"{section_id}_metrics.csv"
    results.to_csv(out, index=False)
    print("Wrote metrics:", out)
    display(results.head())
    return results


def add_deltas_vs_nutrimatch(results: pd.DataFrame) -> pd.DataFrame:
    if results.empty:
        return results
    out = results.copy()
    metric = np.where(out["task_type"].eq("regression"), "r2", "auroc")
    out["primary_metric_name"] = metric
    out["primary_metric"] = np.where(out["task_type"].eq("regression"), out["r2"], out["auroc"])
    base = out[out["arm"].eq("nutrimatch")][["section", "target_id", "model", "task_type", "primary_metric"]].rename(columns={"primary_metric": "nutrimatch_primary_metric"})
    out = out.merge(base, on=["section", "target_id", "model", "task_type"], how="left")
    out["delta_vs_nutrimatch"] = out["primary_metric"] - out["nutrimatch_primary_metric"]
    return out


def plot_best_examples(section_id: str, results: pd.DataFrame, top_n: int = 18) -> pd.DataFrame:
    res = add_deltas_vs_nutrimatch(results)
    if res.empty:
        return res
    enhanced = res[~res["arm"].isin(["age_sex", "base_nutrients", "nutrimatch"])].dropna(subset=["delta_vs_nutrimatch"]).copy()
    if enhanced.empty:
        return enhanced
    best = enhanced.sort_values("delta_vs_nutrimatch", ascending=False).groupby(["target_group", "target_label"], as_index=False).head(1).head(top_n)
    best["example"] = best["target_group"].astype(str) + " | " + best["target_label"].astype(str) + " | " + best["label"].astype(str) + " | " + best["model"].astype(str)
    best.to_csv(OUT_DIR / section_id / f"{section_id}_best_examples_vs_nutrimatch.csv", index=False)

    fig, ax = plt.subplots(figsize=(13, max(5, 0.45 * len(best))))
    sns.barplot(data=best.sort_values("delta_vs_nutrimatch"), x="delta_vs_nutrimatch", y="example", hue="task_type", dodge=False, ax=ax)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(f"{section_id}: best examples vs NutriMatch")
    ax.set_xlabel("Delta primary metric vs NutriMatch (R2 for regression, AUROC for classification)")
    ax.set_ylabel("")
    plt.tight_layout()
    path = FIG_DIR / f"{section_id}_best_examples_vs_nutrimatch.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print("Wrote:", path)
    display(best[["target_group", "target_label", "task", "model", "label", "primary_metric", "nutrimatch_primary_metric", "delta_vs_nutrimatch", "n", "feature_count"]])
    return best


def plot_metric_bars(section_id: str, results: pd.DataFrame) -> None:
    if results.empty:
        return
    res = add_deltas_vs_nutrimatch(results)
    summary = res.groupby(["section", "target_group", "arm", "label", "model", "task_type"], as_index=False).agg(
        mean_primary_metric=("primary_metric", "mean"),
        mean_delta_vs_nutrimatch=("delta_vs_nutrimatch", "mean"),
        targets=("target_id", "nunique"),
    )
    summary.to_csv(OUT_DIR / section_id / f"{section_id}_arm_summary.csv", index=False)
    plot = summary[~summary["arm"].eq("age_sex")].copy()
    fig, ax = plt.subplots(figsize=(14, max(6, 0.35 * plot["label"].nunique())))
    sns.barplot(data=plot, x="mean_delta_vs_nutrimatch", y="label", hue="model", errorbar=None, ax=ax)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(f"{section_id}: mean improvement over NutriMatch")
    ax.set_xlabel("Mean delta primary metric vs NutriMatch")
    ax.set_ylabel("")
    plt.tight_layout()
    path = FIG_DIR / f"{section_id}_mean_delta_vs_nutrimatch.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print("Wrote:", path)
    display(summary.sort_values("mean_delta_vs_nutrimatch", ascending=False).head(30))


def plot_best_scatter_pair(section_id: str, results: pd.DataFrame, targets: pd.DataFrame, prefer_model: str = "ridge") -> None:
    res = add_deltas_vs_nutrimatch(results)
    candidates = res[(~res["arm"].isin(["age_sex", "base_nutrients", "nutrimatch"])) & res["delta_vs_nutrimatch"].notna()].copy()
    if candidates.empty:
        return
    if (candidates["model"] == prefer_model).any():
        candidates = candidates[candidates["model"] == prefer_model]
    best = candidates.sort_values("delta_vs_nutrimatch", ascending=False).iloc[0]
    target_id = best["target_id"]
    model_name = best["model"]
    task_type = best["task_type"]
    best_arm = next(a for a in MODEL_ARMS if a["arm"] == best["arm"])
    nutrimatch_arm = next(a for a in MODEL_ARMS if a["arm"] == "nutrimatch")

    panels = [("NutriMatch", nutrimatch_arm), ("Best alternative: " + best_arm["label"], best_arm)]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
    targets_for_plot = normalize_ids(targets[[ID_COL, target_id]].copy())
    targets_for_plot[ID_COL] = targets_for_plot[ID_COL].astype(str)

    for ax, (title, arm) in zip(axes, panels):
        arm_x = normalize_ids(arm["x"].copy())
        arm_x[ID_COL] = arm_x[ID_COL].astype(str)
        merged = arm_x.merge(targets_for_plot, on=ID_COL, how="inner")
        y = pd.to_numeric(merged[target_id], errors="coerce")
        if task_type == "classification":
            y_model = y.astype("Int64").astype(str)
        else:
            y_model = y
        X = merged.drop(columns=[ID_COL, target_id])
        oof = make_oof_predictions(X, y_model, task_type, model_name)
        if oof is None:
            ax.set_title(title + "\nno OOF predictions")
            continue
        _X2, y2, pred, score = oof
        if task_type == "regression":
            scatter = ax.scatter(y2, pred, c=y2, cmap="viridis", s=20, alpha=0.75)
            lo = min(np.nanmin(y2), np.nanmin(pred)); hi = max(np.nanmax(y2), np.nanmax(pred))
            ax.plot([lo, hi], [lo, hi], color="black", linewidth=1, linestyle="--")
            if model_name == "ridge" and len(y2) > 5:
                coef = np.polyfit(y2.astype(float), np.asarray(pred).astype(float), 1)
                xs = np.linspace(lo, hi, 100)
                ax.plot(xs, coef[0] * xs + coef[1], color="#d95f02", linewidth=2)
            ax.set_xlabel("Observed target")
            ax.set_ylabel("Out-of-fold prediction")
            fig.colorbar(scatter, ax=ax, label="Observed target")
        else:
            if isinstance(score, np.ndarray) and score.ndim == 2:
                score_plot = score[:, -1]
            else:
                score_plot = np.asarray(score)
            y_num = pd.Series(y2).astype("category").cat.codes.to_numpy()
            jitter = np.random.default_rng(RANDOM_STATE).normal(0, 0.04, size=len(y_num))
            scatter = ax.scatter(y_num + jitter, score_plot, c=y_num, cmap="tab10", s=20, alpha=0.75)
            ax.set_xlabel("Observed class")
            ax.set_ylabel("Out-of-fold score/probability")
            fig.colorbar(scatter, ax=ax, label="Observed class")
        ax.set_title(title)
    fig.suptitle(f"{section_id}: {best['target_label']} | {model_name} | {best['task']}")
    plt.tight_layout()
    path = FIG_DIR / f"{section_id}_best_scatter_pair.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print("Wrote:", path)


def section_report(section_id: str, results: pd.DataFrame, targets: pd.DataFrame) -> pd.DataFrame:
    plot_metric_bars(section_id, results)
    best = plot_best_examples(section_id, results)
    try:
        plot_best_scatter_pair(section_id, results, targets)
    except Exception as exc:
        print(f"Scatter-pair plot failed for {section_id}; continuing benchmark. Error: {exc}")
    return best


def section_metrics_path(section_id: str) -> Path:
    return OUT_DIR / section_id / f"{section_id}_metrics.csv"


def load_section_results(section_id: str) -> pd.DataFrame:
    path = section_metrics_path(section_id)
    if path.exists():
        df = pd.read_csv(path, low_memory=False)
        print("Loaded saved metrics:", path, df.shape)
        return df
    print("No saved metrics yet:", path)
    return pd.DataFrame()


def maybe_run_section(section_id: str, targets: pd.DataFrame, catalog: pd.DataFrame) -> pd.DataFrame:
    if RUN_TRAINING:
        return run_benchmark_section(section_id, targets, catalog)
    print(f"Skipping training for {section_id}. Run the background command above, then rerun this notebook to load results.")
    return load_section_results(section_id)


def maybe_section_report(section_id: str, results: pd.DataFrame, targets: pd.DataFrame) -> pd.DataFrame:
    if results is None or results.empty:
        print(f"No metrics available for {section_id}; plots will appear after background training finishes.")
        return pd.DataFrame()
    return section_report(section_id, results, targets)


## 8. Benchmark Section: NutriMatch Paper-Aligned Prediction

This section mirrors the NutriMatch paper comparison as closely as possible here: age/sex, base nutrients, all NutriMatch nutrients, and enhanced features. The target groups include body composition, anthropometry, serum folate/glucose-related blood tests, and CGM traits.


In [ ]:
PAPER_SECTION = "01_nutrimatch_paper_aligned"

paper_table_specs = [
    {"dataset": "body_composition", "table": "body_composition", "target_group": "body_composition", "include_regex": r"body_comp.*fat|total_scan.*vat|visceral|fat_mass|percent_fat", "max_targets": 10},
    {"dataset": "anthropometrics", "table": "anthropometrics", "target_group": "anthropometry", "include_regex": r"waist|hip|bmi|body_mass_index", "max_targets": 8},
    {"dataset": "blood_tests", "table": "blood_tests", "target_group": "blood_biomarkers", "include_regex": r"folate|folic|b9|glucose|hba1c|triglyceride|cholesterol|hdl|ldl|alt|ast|ggt", "max_targets": 12},
    {"dataset": "cgm", "table": "iglu", "target_group": "cgm", "include_regex": r"cgm_mean|cgm_median|cgm_auc|cgm_gmi|cgm_ea1c|cgm_in_range|cgm_above|cgm_grade|cgm_cv|cgm_sd", "max_targets": 12},
]

paper_targets, paper_catalog = load_direct_targets(PAPER_SECTION, paper_table_specs, max_targets=MAX_TARGETS_PER_SECTION)
display(paper_catalog)
paper_results = maybe_run_section(PAPER_SECTION, paper_targets, paper_catalog)
paper_best = maybe_section_report(PAPER_SECTION, paper_results, paper_targets)


## 9. Benchmark Section: Gut Microbiome From Diet

Uses `PhenoLoader('gut_microbiome')`. The knowledgebase says the dataset includes `gut_microbiome.parquet`, URS relative abundance, and MetaPhlAn abundance files by taxonomic level. This cell will use whichever local bulk files are exposed in your TRE mount and cache the resulting target matrix.


In [ ]:
GUT_SECTION = "02_gut_microbiome"

gut_bulk_patterns = [
    ("gut_urs", r"urs"),
    ("gut_metaphlan_species", r"metaphlan.*species.*parquet|metaphlan_abundance_species"),
    ("gut_metaphlan_genus", r"metaphlan.*genus.*parquet|metaphlan_abundance_genus"),
    ("gut_metaphlan_family", r"metaphlan.*family.*parquet|metaphlan_abundance_family"),
]

gut_targets, gut_catalog = load_microbiome_targets(
    GUT_SECTION,
    dataset="gut_microbiome",
    primary_table="gut_microbiome",
    bulk_patterns=gut_bulk_patterns,
    include_regex=r"read_count|shannon|simpson|richness|abundance|akkermansia|bifidobacter|prevotella|bacteroides|firmicutes|microbiome",
    max_targets=MAX_TARGETS_PER_SECTION,
)
display(gut_catalog.head(80))
gut_results = maybe_run_section(GUT_SECTION, gut_targets, gut_catalog)
gut_best = maybe_section_report(GUT_SECTION, gut_results, gut_targets)


## 10. Benchmark Section: Nightingale Metabolomics From Diet

Uses `PhenoLoader('nightingale_metabolomics')`. The knowledgebase says this contains metabolite quantification plus per-biomarker QC tags. This benchmark uses continuous metabolite columns from the `nightingale_metabolomics` table.


In [ ]:
METAB_SECTION = "03_nightingale_metabolomics"

metab_table_specs = [
    {"dataset": "nightingale_metabolomics", "table": "nightingale_metabolomics", "target_group": "nightingale_metabolomics", "include_regex": r".*", "exclude_regex": r"tag|qc|collection|timezone", "max_targets": MAX_TARGETS_PER_SECTION},
]

metab_targets, metab_catalog = load_direct_targets(METAB_SECTION, metab_table_specs, max_targets=MAX_TARGETS_PER_SECTION)
display(metab_catalog.head(80))
metab_results = maybe_run_section(METAB_SECTION, metab_targets, metab_catalog)
metab_best = maybe_section_report(METAB_SECTION, metab_results, metab_targets)


## 11. Benchmark Section: Oral Microbiome From Diet

Uses `PhenoLoader('oral_microbiome')`. This follows the working nitrate/oral-microbiome pattern: start from the main table, then try MetaPhlAn and HUMAnN linked bulk files if local paths are present.


In [ ]:
ORAL_SECTION = "04_oral_microbiome"

oral_bulk_patterns = [
    ("oral_metaphlan_species", r"metaphlan.*species.*parquet|metaphlan_abundance_species"),
    ("oral_metaphlan_genus", r"metaphlan.*genus.*parquet|metaphlan_abundance_genus"),
    ("oral_metaphlan_family", r"metaphlan.*family.*parquet|metaphlan_abundance_family"),
    ("oral_humann_pathway_abundance", r"humann.*pathway.*abundance.*(parquet|arrow)"),
    ("oral_humann_pathway_coverage", r"humann.*pathway.*coverage.*(parquet|arrow)"),
]

oral_targets, oral_catalog = load_microbiome_targets(
    ORAL_SECTION,
    dataset="oral_microbiome",
    primary_table="oral_microbiome",
    bulk_patterns=oral_bulk_patterns,
    include_regex=r"read_count|shannon|simpson|richness|neisseria|rothia|veillonella|prevotella|actinomyces|haemophilus|nitrate|nitrite|nitrogen|pathway|abundance",
    max_targets=MAX_TARGETS_PER_SECTION,
)
display(oral_catalog.head(100))
oral_results = maybe_run_section(ORAL_SECTION, oral_targets, oral_catalog)
oral_best = maybe_section_report(ORAL_SECTION, oral_results, oral_targets)


## 12. Benchmark Section: CGM / Post-Meal Glucose Proxy Traits From Diet

The HPP `cgm` dataset exposes `cgm`, `iglu`, and `iglu_daily` tables. GluFormer-style meal-level forecasting requires raw CGM tokenization and a heavier time-series model; this sklearn benchmark uses the available CGM summary traits as fast post-meal/glucose-response proxies: mean glucose, AUC, GMI/eA1c, glucose variability, and time above/in range thresholds.


In [ ]:
CGM_SECTION = "05_cgm_post_meal_glucose_proxies"

cgm_table_specs = [
    {"dataset": "cgm", "table": "iglu", "target_group": "cgm_iglu", "include_regex": r"cgm_mean|cgm_median|cgm_auc|cgm_gmi|cgm_ea1c|cgm_above|cgm_below|cgm_in_range|cgm_cv|cgm_sd|cgm_mage|cgm_grade|cgm_hbgi|cgm_lbgi|cgm_j_index", "max_targets": 18},
    {"dataset": "cgm", "table": "iglu_daily", "target_group": "cgm_iglu_daily", "include_regex": r"cgm_daily_mean|cgm_daily_median|cgm_daily_auc|cgm_daily_gmi|cgm_daily_ea1c|cgm_daily_above|cgm_daily_below|cgm_daily_in_range|cgm_daily_cv|cgm_daily_sd|cgm_daily_mage|cgm_daily_grade", "max_targets": 18},
]

cgm_targets, cgm_catalog = load_direct_targets(CGM_SECTION, cgm_table_specs, max_targets=MAX_TARGETS_PER_SECTION)
display(cgm_catalog.head(100))
cgm_results = maybe_run_section(CGM_SECTION, cgm_targets, cgm_catalog)
cgm_best = maybe_section_report(CGM_SECTION, cgm_results, cgm_targets)


## 13. Benchmark Section: Peripheral Vascular Health From Diet

Uses `PhenoLoader('vascular_health')`, which is the loader name shown in the TRE screenshots for the peripheral vascular health dataset. Targets include ankle/brachial pressures, ABI, and pulse wave velocity.


In [ ]:
VASC_SECTION = "06_peripheral_vascular_health"

vascular_table_specs = [
    {"dataset": "vascular_health", "table": "vascular_health", "target_group": "vascular_health", "include_regex": r"ankle_pressure|brachial_pressure|abi|pwv", "max_targets": 20},
]

vascular_targets, vascular_catalog = load_direct_targets(VASC_SECTION, vascular_table_specs, max_targets=MAX_TARGETS_PER_SECTION)
display(vascular_catalog)
vascular_results = maybe_run_section(VASC_SECTION, vascular_targets, vascular_catalog)
vascular_best = maybe_section_report(VASC_SECTION, vascular_results, vascular_targets)


## 14. Combined Benchmark Summary

This combines all sections into one table and highlights where enhanced diet features beat NutriMatch.


In [ ]:
SECTION_IDS = [
    PAPER_SECTION,
    GUT_SECTION,
    METAB_SECTION,
    ORAL_SECTION,
    CGM_SECTION,
    VASC_SECTION,
]

result_frames = []
for section_id in SECTION_IDS:
    frame = globals().get(section_id.split("_", 1)[0] + "_results")
    if isinstance(frame, pd.DataFrame) and not frame.empty:
        result_frames.append(frame)
    else:
        loaded = load_section_results(section_id)
        if not loaded.empty:
            result_frames.append(loaded)

if not result_frames:
    print("No saved section metrics yet. Start the console/background run, wait for it to finish, then rerun this notebook.")
    all_results = pd.DataFrame()
    metric_table = pd.DataFrame()
    summary = pd.DataFrame()
    best_by_target = pd.DataFrame()
    best_overall = pd.DataFrame()
else:
    all_results = pd.concat(result_frames, ignore_index=True)
    all_results = add_deltas_vs_nutrimatch(all_results)

    combined_path = OUT_DIR / "diet_data_enhancement_benchmark_I_all_metrics.csv"
    all_results.to_csv(combined_path, index=False)
    print("Wrote combined metrics:", combined_path)

    metric_cols = [
        "section", "target_group", "target_label", "dataset", "table", "column",
        "task", "model", "label", "arm", "n", "feature_count",
        "r2", "rmse", "pearson_r", "accuracy", "balanced_accuracy", "f1_macro", "auroc", "auprc",
        "primary_metric_name", "primary_metric", "nutrimatch_primary_metric", "delta_vs_nutrimatch",
    ]
    metric_cols = [c for c in metric_cols if c in all_results.columns]
    metric_table = all_results[metric_cols].copy()
    metric_table_path = OUT_DIR / "diet_data_enhancement_benchmark_I_metric_table.csv"
    metric_table.to_csv(metric_table_path, index=False)
    print("Wrote metric table:", metric_table_path)

    summary = all_results.groupby(["section", "arm", "label", "model", "task", "task_type"], as_index=False).agg(
        targets=("target_id", "nunique"),
        mean_r2=("r2", "mean"),
        mean_rmse=("rmse", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_auroc=("auroc", "mean"),
        mean_auprc=("auprc", "mean"),
        mean_primary_metric=("primary_metric", "mean"),
        mean_delta_vs_nutrimatch=("delta_vs_nutrimatch", "mean"),
        wins_vs_nutrimatch=("delta_vs_nutrimatch", lambda s: int((s > 0).sum())),
    )
    summary_path = OUT_DIR / "diet_data_enhancement_benchmark_I_summary.csv"
    summary.to_csv(summary_path, index=False)
    print("Wrote summary:", summary_path)

    best_by_target = (
        all_results.dropna(subset=["primary_metric"])
        .sort_values("primary_metric", ascending=False)
        .groupby(["section", "target_id", "model", "task_type"], as_index=False)
        .head(1)
    )
    best_by_target_path = OUT_DIR / "diet_data_enhancement_benchmark_I_best_by_target.csv"
    best_by_target.to_csv(best_by_target_path, index=False)
    print("Wrote best by target:", best_by_target_path)

    best_overall = (
        all_results[~all_results["arm"].isin(["age_sex", "base_nutrients", "nutrimatch"])]
        .dropna(subset=["delta_vs_nutrimatch"])
        .sort_values("delta_vs_nutrimatch", ascending=False)
        .head(40)
    )
    best_overall_path = OUT_DIR / "diet_data_enhancement_benchmark_I_best_overall.csv"
    best_overall.to_csv(best_overall_path, index=False)
    print("Wrote best overall:", best_overall_path)

    print("\nMetric table: which data gives what accuracy for each task")
    display(metric_table.sort_values(["section", "target_group", "target_label", "model", "label"]).head(300))
    print("\nArm summary")
    display(summary.sort_values(["section", "mean_delta_vs_nutrimatch"], ascending=[True, False], na_position="last").head(150))
    print("\nBest data source per target/model")
    display(best_by_target[[c for c in ["section", "target_group", "target_label", "task", "model", "label", "primary_metric_name", "primary_metric", "n", "feature_count"] if c in best_by_target.columns]].head(150))
    print("\nBest examples supporting the enhancement thesis")
    display(best_overall[[c for c in ["section", "target_group", "target_label", "task", "model", "label", "primary_metric", "nutrimatch_primary_metric", "delta_vs_nutrimatch", "n", "feature_count"] if c in best_overall.columns]])

    fig, ax = plt.subplots(figsize=(14, max(6, 0.35 * len(summary["label"].unique()))))
    plot_summary = summary[~summary["arm"].eq("age_sex")].copy()
    sns.barplot(data=plot_summary, x="mean_delta_vs_nutrimatch", y="section", hue="label", errorbar=None, ax=ax)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title("Diet Data Enhancement Benchmark I: mean delta vs NutriMatch by section")
    ax.set_xlabel("Mean delta primary metric vs NutriMatch")
    ax.set_ylabel("")
    ax.legend(title="Model arm", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    path = FIG_DIR / "diet_data_enhancement_benchmark_I_combined_delta.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print("Wrote:", path)


## 15. Notes For Interpretation

- Treat Ridge and Random Forest as complementary sanity checks. If an improvement appears in both, it is more credible.
- The main comparison is each enhanced arm vs `nutrimatch`, not vs age/sex or base nutrients.
- Classification targets created from continuous targets are convenience benchmarks. They are useful for signal discovery, but the regression target remains the cleaner endpoint unless the threshold is clinically established.
- For CGM, this notebook uses fast summary traits from `iglu`/`iglu_daily`. A true GluFormer-like post-meal glucose experiment would require raw CGM time series, meal timestamps, tokenization, and a heavier temporal model.
- Because some enriched Diet Data Enhancement tables are duplicates or near-duplicates, identical results across some arms are expected and should not be counted as independent evidence.
